# 二进制数据与类型化数组

学习目标：能用缓冲区与视图处理字节数据，明确共享、字节序、调整和分离的边界。

前置知识：数值与 BigInt、数组、对象引用、异常处理。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/27-binary-data/。

1. [views.mjs](scripts/27-binary-data/views.mjs)：视图共享、复制与数值转换。
2. [endian.mjs](scripts/27-binary-data/endian.mjs)：混合字段与显式大端读取。
3. [resize-transfer.mjs](scripts/27-binary-data/resize-transfer.mjs)：长度跟踪、越界恢复与分离。
4. [shared.mjs](scripts/27-binary-data/shared.mjs)：共享视图与原子更新接口。

Step 1：运行缓冲区视图。

```bash
node scripts/27-binary-data/views.mjs
```

Step 2：运行协议字节序示例。

```bash
node scripts/27-binary-data/endian.mjs
```

Step 3：运行调整与转移示例。

```bash
node scripts/27-binary-data/resize-transfer.mjs
```

Step 4：运行共享内存接口示例。

```bash
node scripts/27-binary-data/shared.mjs
```

## 1 缓冲区与视图的分工

先按字节偏移找到视图的覆盖范围，再把视图索引换回缓冲区位置。

ArrayBuffer 表示字节存储，不通过普通数组索引直接读写数据。TypedArray（类型化数组）按一种数值元素格式解释缓冲区，DataView 可在不同字节偏移读写不同格式。TypedArray 是一类构造器的统称，不能直接写 new TypedArray。

视图的 length 是元素数，byteLength 是字节数，byteOffset 是视图起点相对缓冲区的字节偏移。多个视图可共享同一个缓冲区，因此修改不等于复制。构造多字节类型视图时，起始偏移须按元素字节数对齐；DataView 不要求这种元素对齐。

| 类型化数组名称 | 中文名称／含义 | 每元素字节数 |
| --- | --- | --- |
| Int8Array | 8 位有符号整数 | 1 |
| Uint8Array | 8 位无符号整数 | 1 |
| Uint8ClampedArray | 钳制到 0–255 的无符号整数 | 1 |
| Int16Array | 16 位有符号整数 | 2 |
| Uint16Array | 16 位无符号整数 | 2 |
| Int32Array | 32 位有符号整数 | 4 |
| Uint32Array | 32 位无符号整数 | 4 |
| Float16Array | 16 位浮点数 | 2 |
| Float32Array | 32 位浮点数 | 4 |
| Float64Array | 64 位浮点数 | 8 |
| BigInt64Array | 64 位有符号 BigInt | 8 |
| BigUint64Array | 64 位无符号 BigInt | 8 |

Float16Array 属于本课 ECMAScript 2025 基线，Node.js 24 已启用；它存储精度较低，不是任意 Number 的无损容器。整数视图的数值转换和溢出规则不同于范围校验，业务输入需要显式检查。

![多个视图观察同一缓冲区。每格一个字节；本例使用 Uint8Array，所以元素长度与字节长度数值相同。](image/illustration/27-01-buffer-shared-views.svg)

图示说明：图定格在 tail[0] = 70 之后；只画本例的 8 位视图，不暗示多字节类型的字节序。

下面把 tail.byteOffset、tail.length 与图中范围对照；随后修改 shared[0]，观察 copied 为什么不跟着变化。

配套 [views.mjs](scripts/27-binary-data/views.mjs)：

```javascript
import assert from "node:assert/strict";
// 1. buffer 持有字节，两个视图以不同起点观察同一块存储。
const buffer = new ArrayBuffer(8);
const bytes = new Uint8Array(buffer);
const tail = new Uint8Array(buffer, 2, 3);
tail[0] = 70;
assert.equal(bytes[2], 70);
console.log(tail.length, tail.byteLength, tail.byteOffset); // → 3 3 2
// 2. subarray 共享原字节，slice 复制字节；修改前者用于比较两种结果。
const shared = bytes.subarray(2, 4);
const copied = bytes.slice(2, 4);
shared[0] = 80;
assert.equal(bytes[2], 80);
assert.equal(copied[0], 70);
console.log("shared", bytes[2], "copied", copied[0]); // → shared 80 copied 70
// 3. 同样的数值按不同元素类型写入，会采用不同的转换规则。
console.log(new Uint8Array([257, -1]).join(",")); // → 1,255；按无符号 8 位转换
console.log(new Uint8ClampedArray([257, -1]).join(",")); // → 255,0；钳制到范围内
assert.equal(new Float16Array([1.5])[0], 1.5);
assert.equal(new BigInt64Array([12n])[0], 12n);
assert.throws(() => new BigInt64Array([12]), TypeError);
assert.throws(() => new Uint16Array(buffer, 1), RangeError);
console.log("numeric types and alignment checked"); // → numeric types and alignment checked
```

## 2 DataView 与字节序

字节序（endianness）决定多字节数值的字节排列。大端序把高位字节放在较低地址，小端序把低位字节放在较低地址。TypedArray 多字节元素按宿主字节序解释，不能拿它直接假定网络协议格式。

DataView 的多字节 get/set 方法接受 littleEndian 布尔参数；省略或 false 表示大端，true 表示小端。下面定义一个 4 字节小协议：偏移 0 是标记字节，偏移 1–2 是大端 16 位长度，偏移 3 是状态。每个偏移单位都是字节；越出视图范围会抛 RangeError。

配套 [endian.mjs](scripts/27-binary-data/endian.mjs)：

```javascript
import assert from "node:assert/strict";
const buffer = new ArrayBuffer(4);
const view = new DataView(buffer);
view.setUint8(0, 0x7f);
view.setUint16(1, 0x1234, false);
view.setUint8(3, 1);
console.log([...new Uint8Array(buffer)].map((value) => value.toString(16).padStart(2, "0")).join(" "));
// → 7f 12 34 01
assert.equal(view.getUint16(1, false), 0x1234);
assert.equal(view.getUint16(1, true), 0x3412);
assert.throws(() => view.getUint32(1), RangeError);
console.log("big", view.getUint16(1), "little", view.getUint16(1, true)); // → big 4660 little 13330
```

## 3 调整长度与分离缓冲区

创建 ArrayBuffer 时指定 maxByteLength 可得到可调整缓冲区，再用 resize 改变 byteLength，不能超过上限。增长部分初始化为零，缩短会丢弃尾部字节。未显式给长度的视图可跟踪底层长度；固定长度视图在缩短后可能越界，其长度等访问结果要按具体 API 判断。

transfer 将内容交给新的缓冲区并分离原缓冲区；transferToFixedLength 还让目标成为固定长度。分离（detach）不只是把当前视图变量改成新值：所有仍引用旧缓冲区的视图都受影响。不要继续使用旧视图或缓存它的长度。

配套 [resize-transfer.mjs](scripts/27-binary-data/resize-transfer.mjs)：

```javascript
import assert from "node:assert/strict";
const buffer = new ArrayBuffer(4, { maxByteLength: 8 });
// 未指定长度的视图跟踪缓冲区；显式给出 4 的视图固定长度。
const tracking = new Uint8Array(buffer);
const fixed = new Uint8Array(buffer, 0, 4);
tracking.set([1, 2, 3, 4]);
// 先扩展，再缩短到固定视图越界，最后重新扩展观察长度与数据。
buffer.resize(6);
assert.equal(tracking.length, 6);
assert.equal(fixed.length, 4);
assert.equal(tracking[5], 0);
buffer.resize(2);
assert.equal(fixed.length, 0);
assert.equal(tracking.length, 2);
buffer.resize(4);
assert.equal(fixed.length, 4);
console.log(tracking.join(",")); // → 1,2,0,0；被截断字节不会恢复
assert.throws(() => buffer.resize(9), RangeError);

// transfer 后改用新缓冲区；旧缓冲区及其视图已不能继续访问原字节。
const view = new DataView(buffer);
const moved = buffer.transferToFixedLength();
assert.equal(buffer.detached, true);
assert.equal(tracking.length, 0);
assert.throws(() => view.getUint8(0), TypeError);
assert.equal(moved.resizable, false);
console.log("moved", new Uint8Array(moved).join(",")); // → moved 1,2,0,0
// 对照 transfer：它保留本例缓冲区的可调整属性。
const growable = new ArrayBuffer(2, { maxByteLength: 4 });
const transferred = growable.transfer();
assert.equal(transferred.resizable, true);
console.log("detached and resize policies checked"); // → detached and resize policies checked
```

## 4 共享内存与 Atomics 的定位

SharedArrayBuffer 可使多个 Agent 访问同一份共享字节；它与把普通 ArrayBuffer 转移给另一个环境不同，不能像普通缓冲区那样通过 transfer 分离。普通的读—改—写组合不能自动成为一个原子操作。

Atomics 提供适用于指定整数视图的原子读取、写入及读改写方法。Atomics.add 返回修改前的值，compareExchange 只有旧值匹配时才替换。本例只验证单 Agent 中的接口语义，不把一次结果当作并发正确性的证明。实际 Worker 协调、消息协议和等待策略留在运行时专题。

浏览器共享内存受安全上下文与跨源隔离等宿主条件约束，不能因为 Node.js 可用就认为任意网页可用。Node.js Worker 可共享 SharedArrayBuffer；浏览器主线程也不能随意使用会阻塞的 Atomics.wait。

配套 [shared.mjs](scripts/27-binary-data/shared.mjs)：

```javascript
import assert from "node:assert/strict";
const buffer = new SharedArrayBuffer(Int32Array.BYTES_PER_ELEMENT);
const counts = new Int32Array(buffer);
const alias = new Int32Array(buffer);
assert.equal(Atomics.store(counts, 0, 5), 5);
assert.equal(Atomics.add(counts, 0, 2), 5);
assert.equal(Atomics.load(alias, 0), 7);
assert.equal(Atomics.compareExchange(counts, 0, 7, 10), 7);
assert.equal(Atomics.compareExchange(counts, 0, 7, 99), 10);
console.log("shared atomic value", Atomics.load(alias, 0)); // → shared atomic value 10
```

## 本章小结

- 缓冲区保存字节，视图决定解释方式；共享视图、复制视图与转移所有权要分清。
- 协议字段用 DataView 显式指定字节序与偏移。
- 调整、分离与共享内存各有不同规则；原子 API 不替代完整并发协议。

## 练习

1. 把大端长度改为 0x0102；标准：偏移 1–2 的字节为 01 02，小端误读得到 513。
2. 缩短缓冲区后再增长；标准：保留区间不变，重新增长的尾部为零，旧数据不复活。
3. 让 compareExchange 的期望值不匹配；标准：返回当前值且缓冲区不被修改。

## 参考与引用来源

- TC39（ECMA-262 第 16 版）：[§25.1–25.4 缓冲区、DataView 与 Atomics](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-arraybuffer-objects)，尤其 resize、transfer、ArrayBufferCopyAndDetach、GetViewValue；[§23.2 TypedArray](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-typedarray-objects)：元素类型、长度、共享视图及复制。
- Node.js：[24.0.0 发布说明](https://nodejs.org/en/blog/release/v24.0.0)中的 Float16Array；24.11.0 [Worker 传输与共享缓冲区](https://nodejs.org/download/release/v24.11.0/docs/api/worker_threads.html#considerations-when-transferring-typedarrays-and-buffers)：宿主共享条件。
- WHATWG：[HTML Agent cluster 与共享内存](https://html.spec.whatwg.org/multipage/webappapis.html#integration-with-the-javascript-agent-cluster-formalism)：网页跨源隔离与 Agent 的限制。